# Resume ↔ Job Description Matcher

A beginner-friendly project that does something useful: given a resume and some
job descriptions, rank the jobs by fit and show which skills are missing.

**Two ideas, layered:**

1. **Keyword matching** — find named skills in both texts, report the overlap.
   Simple, explainable, and produces the genuinely useful output (the skill gap).
2. **Semantic similarity** — embed both texts and compare. Catches phrasing that
   keyword matching misses: *"built ML pipelines"* vs *"experience with machine
   learning workflows"* share almost no words but mean nearly the same thing.

**What makes this more than a demo:** section 6 evaluates the matcher against a
real labelled dataset. Almost every resume-matcher project skips this and just
shows a pretty UI. The evaluation is the part that makes it defensible.

---

### Run order

Cells run top to bottom. Section 6 downloads a dataset — everything before it
works offline once the model is cached.

## 1. Setup

In [ ]:
!pip install -q sentence-transformers scikit-learn pandas numpy

import re, numpy as np, pandas as pd
print("ready")

## 2. Skill vocabulary

A hand-written list of skills and the ways people write them. Deliberately not a
model: it's explainable, easy to debug, and you can defend every entry.

Keep it short here — the full version lives in `skills_vocab.py`.

In [ ]:
SKILLS = {
    "Python":       ["python"],
    "Java":         ["java"],
    "C":            ["c"],
    "C++":          ["c++", "cpp"],
    "C#":           ["c#", "csharp"],
    "JavaScript":   ["javascript", "js"],
    "R":            ["r"],
    "SQL":          ["sql"],
    "Machine Learning": ["machine learning", "ml"],
    "Deep Learning":    ["deep learning"],
    "NLP":          ["nlp", "natural language processing"],
    "PyTorch":      ["pytorch", "torch"],
    "TensorFlow":   ["tensorflow"],
    "Scikit-learn": ["scikit-learn", "sklearn", "scikit learn"],
    "HuggingFace":  ["huggingface", "hugging face", "transformers"],
    "Pandas":       ["pandas"],
    "NumPy":        ["numpy"],
    "Power BI":     ["power bi", "powerbi"],
    "Tableau":      ["tableau"],
    "Excel":        ["excel"],
    "Docker":       ["docker"],
    "AWS":          ["aws", "amazon web services"],
    "Git":          ["git", "github"],
    "Linux":        ["linux"],
    "React":        ["react", "reactjs"],
    "Node.js":      ["node.js", "nodejs"],
    "PostgreSQL":   ["postgresql", "postgres"],
    "Kubernetes":   ["kubernetes", "k8s"],
    "REST API":     ["rest api", "restful"],
    "CI/CD":        ["ci/cd", "cicd"],
    "Communication":["communication skills"],
    "Data Visualization": ["data visualization", "data visualisation"],
}
print(len(SKILLS), "skills")

## 3. Skill extraction — and the bug that will bite you

Naive substring search is wrong. `"c" in "c++"` is `True`, so every C++ developer
gets credited with C as well. Silent, and it inflates your numbers.

`\b` word boundaries don't fix it either, because `+` is already a non-word
character — so `\bc\b` still matches the `c` in `c++`.

The fix is an explicit boundary class that includes `+` and `#`. Note it does
**not** include `/`, because people write `R/Python` and we want that to match R.

**Write the tests.** This is a five-line function with three edge cases, which is
exactly the size of thing people don't test and then get wrong.

In [ ]:
def clean_text(text):
    if not text:
        return ""
    text = re.sub(r"[^\w\s+#/.\-]", " ", text.lower())
    return re.sub(r"\s+", " ", text).strip()

def _pattern(form):
    # boundary class includes + and # so "c" doesn't match inside "c++" / "c#"
    return re.compile(rf"(?<![a-z0-9+#]){re.escape(form)}(?![a-z0-9+#])")

_COMPILED = {k: [_pattern(f) for f in v] for k, v in SKILLS.items()}

def extract_skills(text):
    t = clean_text(text)
    return {k for k, pats in _COMPILED.items() if any(p.search(t) for p in pats)}

In [ ]:
# --- tests: (text, must contain, must NOT contain) ---
cases = [
    ("I know C++ and C#",      {"C++", "C#"},          {"C"}),
    ("Python, Java, C",        {"C", "Python", "Java"}, set()),
    ("R/Python for analysis",  {"R", "Python"},         set()),
    ("we use ci/cd pipelines", {"CI/CD"},               set()),
    ("javascript only",        {"JavaScript"},          {"Java"}),
    ("scarce resources",       set(),                   {"R"}),
    ("carpentry work",         set(),                   {"C"}),
]

failures = 0
for text, must, must_not in cases:
    got = extract_skills(text)
    ok = must.issubset(got) and not (must_not & got)
    failures += (not ok)
    print(f"{'OK  ' if ok else 'FAIL'} {text!r:26s} -> {sorted(got)}")

assert failures == 0, f"{failures} test(s) failed"
print("\nall extraction tests passed")

## 4. Sample data

In [ ]:
resume = '''
Abhishikta Dutta - Computer Science Engineering student.
Programming: Python, Java, C.
ML/DL: PyTorch, HuggingFace Transformers, Scikit-learn.
Data: Pandas, NumPy, Matplotlib, Seaborn.
Fine-tuned BERT and DistilBERT for misinformation classification on LIAR2.
Built end-to-end ML pipelines with custom Dataset classes and evaluation suites.
Tools: Git, Linux, Jupyter, Google Colab.
'''

jobs = {
    "ML Engineer Intern":
        "Seeking Python and PyTorch experience. Deep learning and NLP background "
        "required. Familiarity with Docker and AWS a plus. You will fine-tune "
        "transformer models and build training pipelines.",
    "Data Analyst":
        "Requires strong SQL, Excel, Power BI and Tableau. Communication skills "
        "essential. You will build dashboards and present findings to stakeholders. "
        "Data visualization experience needed.",
    "Backend Developer":
        "Node.js, React, PostgreSQL, REST API design. Docker and Kubernetes for "
        "deployment. CI/CD pipeline experience preferred.",
}

for name, jd in jobs.items():
    print(f"{name:22s} {sorted(extract_skills(jd))}")

## 5a. Method 1 — TF-IDF similarity

TF-IDF weights words by how distinctive they are, then we take the cosine angle
between documents. Fast, no model download, no GPU.

Its weakness is total: **it can only match words that literally appear in both
texts.** Watch what it does to the jobs that use different vocabulary.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def tfidf_similarity(resume, jds):
    docs = [clean_text(resume)] + [clean_text(j) for j in jds]
    X = TfidfVectorizer(stop_words="english", ngram_range=(1, 2)).fit_transform(docs)
    return cosine_similarity(X[0:1], X[1:]).ravel()

names = list(jobs); jds = list(jobs.values())
tf = tfidf_similarity(resume, jds)
for n, s in sorted(zip(names, tf), key=lambda x: -x[1]):
    print(f"  {n:22s} {s:.4f}")

**Look at the scores that came out as 0.0000.**

That is not a bug. It means the resume and that job description share literally
no vocabulary after stop-word removal. TF-IDF has no way to know that a person
who has done "deep learning" might handle "data visualization" — it sees
unrelated strings.

This is the honest motivation for embeddings, and it is worth stating in your
README exactly this way. You didn't add embeddings because they're fashionable;
you added them because you hit a specific failure and can show it.

## 5b. Method 2 — embedding similarity

A sentence embedding model maps text to a vector where similar *meaning* lands
in a similar place, regardless of the exact words. First run downloads ~90MB.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embedding_similarity(resume, jds, model=model):
    emb = model.encode([resume] + list(jds), normalize_embeddings=True,
                       convert_to_numpy=True)
    return (emb[0:1] @ emb[1:].T).ravel()   # normalised -> dot product = cosine

em = embedding_similarity(resume, jds)
pd.DataFrame({"job": names, "tfidf": tf.round(4), "embedding": em.round(4)}) \
  .sort_values("embedding", ascending=False).reset_index(drop=True)

### The most important thing on this page

Look at the embedding column. The *unrelated* jobs still score well above zero —
often 0.2 to 0.5, sometimes higher.

**So a raw embedding score is not a "match percentage."** If you display
`0.61 → "61% match"` in your app, that number is meaningless: an unrelated job
would also show ~50%. Nearly every resume-matcher project on GitHub does exactly
this, and it's wrong.

What the score *is* good for is **ranking**. "This job suits you better than that
one" is a claim the number genuinely supports.

The app is built around ranking for this reason. Not showing a fake percentage is
a small decision that demonstrates you understood your own metric — and it is the
kind of thing an interviewer will actually probe.

## 5c. Skill gap — the actually useful output

Similarity gives you a ranking. The skill gap gives the user something to *do*.

In [ ]:
def skill_gap(resume, jd):
    r, j = extract_skills(resume), extract_skills(jd)
    return {
        "matched": sorted(j & r),
        "missing": sorted(j - r),
        "coverage": len(j & r) / len(j) if j else 0.0,
    }

for n in names:
    g = skill_gap(resume, jobs[n])
    print(f"\n{n}  —  coverage {g['coverage']:.0%}")
    print(f"   have   : {', '.join(g['matched']) or '-'}")
    print(f"   missing: {', '.join(g['missing']) or '-'}")

Unlike the similarity score, **coverage is a real fraction** — "you have 2 of
the 6 skills this job names" means exactly what it says. Report it with confidence.

That contrast (one metric interpretable, one only comparative) is worth a
paragraph in your README.

### Now look at what it got wrong

The ML Engineer role probably shows **Deep Learning** and **NLP** as *missing* —
even though the resume is obviously a deep learning resume.

Why? The resume says `ML/DL:` and the job says `deep learning`. Keyword matching
sees two different strings. The candidate is penalised for abbreviating.

This is the exact weakness embeddings are meant to cover, and it's also a real
warning about automated resume screening: **write the words out in full**, because
a lot of real screening systems are no smarter than this one.

Both of those observations belong in your README. Finding a concrete failure in
your own tool and explaining it is worth more than another feature.

## 6. Evaluation — does any of this actually work?

Everything so far is anecdote. Three sample jobs where the answer was obvious
proves nothing.

Here we use a public dataset of resume / job-description pairs labelled by
fit, and ask one question:

> **Do "Good Fit" pairs score higher than "No Fit" pairs?**

If they don't, the matcher doesn't work, and you need to know that before you
put it on your portfolio.

> ⚠️ I verified this dataset exists on Hugging Face and that its labels are
> "No Fit" / "Potential Fit" / "Good Fit". I could **not** verify the exact
> column names or split names, so the cell below prints the structure first and
> you may need to adjust the column names in the next cell. That is normal.

In [ ]:
from datasets import load_dataset

ds = load_dataset("cnamuangtoun/resume-job-description-fit")
print(ds)
print("\ncolumns:", ds[list(ds.keys())[0]].column_names)
print("\nfirst row:")
first = ds[list(ds.keys())[0]][0]
for k, v in first.items():
    preview = str(v)[:200].replace("\n", " ")
    print(f"  {k}: {preview}...")

In [ ]:
# --- adjust these three names if the printout above disagrees ---
SPLIT      = "test" if "test" in ds else list(ds.keys())[0]
RESUME_COL = "resume_text"
JD_COL     = "job_description_text"
LABEL_COL  = "label"

data = ds[SPLIT]
cols = data.column_names

def pick(preferred, keywords):
    if preferred in cols:
        return preferred
    for c in cols:
        if any(k in c.lower() for k in keywords):
            return c
    raise KeyError(f"no column matching {keywords} in {cols}")

RESUME_COL = pick(RESUME_COL, ["resume"])
JD_COL     = pick(JD_COL, ["job", "descript"])
LABEL_COL  = pick(LABEL_COL, ["label", "fit"])
print("using columns:", RESUME_COL, "|", JD_COL, "|", LABEL_COL)

N = 300   # keep it small so this runs in a couple of minutes
sample = data.select(range(min(N, len(data))))
print(f"evaluating on {len(sample)} pairs")
print(pd.Series(sample[LABEL_COL]).value_counts())

In [ ]:
# Score every pair with both methods
resumes = sample[RESUME_COL]
jd_list = sample[JD_COL]
labels  = sample[LABEL_COL]

emb_r = model.encode(list(resumes), normalize_embeddings=True,
                     convert_to_numpy=True, batch_size=32, show_progress_bar=True)
emb_j = model.encode(list(jd_list), normalize_embeddings=True,
                     convert_to_numpy=True, batch_size=32, show_progress_bar=True)
emb_scores = (emb_r * emb_j).sum(axis=1)          # row-wise cosine

cov_scores = np.array([skill_gap(r, j)["coverage"] for r, j in zip(resumes, jd_list)])

df = pd.DataFrame({"label": labels, "embedding": emb_scores, "coverage": cov_scores})
df.groupby("label")[["embedding", "coverage"]].agg(["mean", "std", "count"]).round(4)

### Read that table carefully

If "Good Fit" has a higher mean than "No Fit", the signal is real. **But check the
standard deviations too** — if the means differ by 0.02 and the std is 0.15, the
distributions overlap almost completely and the difference is close to useless
for ranking any individual pair.

A separation you can see in a group mean but not use on a single case is a
common and honest finding. Report it as such.

In [ ]:
from sklearn.metrics import roc_auc_score

# Binary: Good Fit vs No Fit (drop the middle class for a clean two-class AUC)
lab = pd.Series(labels).astype(str)
mask = lab.str.contains("good", case=False) | lab.str.contains("no", case=False)
y = lab[mask].str.contains("good", case=False).astype(int).values

print(f"{mask.sum()} pairs after dropping the middle class "
      f"({y.sum()} good fit, {(1-y).sum()} no fit)\n")

for name, scores in [("embedding similarity", emb_scores), ("skill coverage", cov_scores)]:
    s = np.asarray(scores)[mask.values]
    if len(np.unique(y)) < 2:
        print(f"{name:22s} — only one class present, AUC undefined")
        continue
    auc = roc_auc_score(y, s)
    print(f"{name:22s} AUC = {auc:.3f}")

print("\n0.50 = no better than a coin flip.  0.70+ = genuinely useful.")
print("Whatever you get, put it in your README. A modest honest number beats")
print("no number at all, and it is the thing that makes this a project rather")
print("than a demo.")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col, title in zip(axes, ["embedding", "coverage"],
                          ["Embedding similarity", "Skill coverage"]):
    for lv in df["label"].unique():
        ax.hist(df.loc[df["label"] == lv, col], bins=25, alpha=0.55, label=str(lv))
    ax.set_title(title); ax.set_xlabel("score"); ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("count")
plt.suptitle("Do the distributions actually separate?")
plt.tight_layout(); plt.show()

**The overlap in these histograms is your real result.** Two bumps that sit almost
on top of each other means the method barely works, no matter what the mean
table said. Screenshot this for your README either way.

---

## 7. Where to go next

In rough order of value:

1. **Grow the skill vocabulary.** Most missed matches will trace back to a skill
   that simply isn't in the list. Look at ten failures and see.
2. **Try a better embedding model.** Swap in `BAAI/bge-small-en-v1.5` and re-run
   section 6. Does the AUC move? Note that BGE expects an instruction prefix on
   the query side — read its model card, since getting this wrong quietly costs
   you performance.
3. **Combine the two signals.** Fit a simple logistic regression on
   `[embedding_score, skill_coverage]` and see whether it beats either alone.
   Nice and small, and it's a real modelling decision you can talk about.
4. **Chunk long resumes.** Embedding models truncate at 512 tokens, so the back
   half of a two-page resume is currently being ignored entirely. Split into
   sections, embed each, take the max similarity per JD requirement.
5. **Deploy it.** Streamlit Community Cloud is free and takes about ten minutes.
   A live link on your portfolio card is worth a lot more than a screenshot.

## 8. What to write in the README

Lead with the evaluation result, not the feature list:

> *"Ranks job descriptions against a resume using sentence embeddings and
> keyword-based skill extraction. Evaluated on N labelled resume-JD pairs:
> AUC X.XX for embedding similarity, Y.YY for skill coverage. Reports a ranking
> rather than a match percentage, because raw cosine similarity is not
> calibrated — unrelated pairs still score ~0.5."*

That last sentence is the one that will make someone read the rest.